In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Load the dataset
loading the datasets and check the columns, informations

In [ ]:
df=pd.read_csv("retail_store_inventory.csv")
df.describe()
df.info()
df.head()


# Handling dataset
I check for missing values and duplicate values and after that I check whether the value are correct or not and for the "Demand Forecast" feature I get negative values which are incorrect so I try to use max of 0 and itself.

In [ ]:
#handling missing ,duplicate value and check correct values
print("Check missing values")
print(df.isnull().sum())

print("Check duplicate values")
print(df.duplicated().sum())
df["Demand Forecast"]=df["Demand Forecast"].apply(lambda x:max(x,0))
df.info()
df.isnull().sum()

# Feature Engineering
I created time based features from date and lagging and rolling features to capture past behaviour and predict feature

In [ ]:
#feature engneering for time based
df["Date"]=pd.to_datetime(df["Date"])

df["Year"]=df["Date"].dt.year
df["Quarter"]=df["Date"].dt.quarter
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df["Day"]=df['Date'].dt.day

df['Weekday'] = df['Date'].dt.day_name()
df["Is_Weekend"]=df["Date"].dt.weekday.apply(lambda x:1 if x>=5 else 0)

def Season(month):
    if month in [12,1,2]:
        return 'Winter'
    elif month in [3,4,5]:
        return 'Spring'
    elif month in [6,7,8]:
        return 'Summer'
    else:
        return 'Autumn'
df["Season"]=df['Month'].apply(Season)



In [ ]:
#lagging feature to capute past behaviour and to predict future
df_original=df.copy()
df = df_original.copy()  # use the copy of  your original dataset
#we sort the data for chronological order
df=df.sort_values(["Store ID","Product ID","Date"])
df=df.reset_index(drop=True)


df["Sales_lag_1"]=df.groupby(["Store ID","Product ID"])['Units Sold'].shift(1)
df["Sales_lag_7"]=df.groupby(["Store ID","Product ID"])["Units Sold"].shift(7)
df["Sales_lag_30"]=df.groupby(["Store ID","Product ID"])["Units Sold"].shift(30)

df['Rolling_mean_3'] = df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['Rolling_std_3'] = df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(3, min_periods=1).std())

# Binary features for events
df['Is_Holiday'] = df['Holiday/Promotion'].apply(lambda x: 1 if x>0 else 0)

# Lag differences to capture sudden changes and changes for larger period
df['Sales_lag_1_diff'] = df["Units Sold"] - df['Sales_lag_1']
df['Sales_lag_1_vs_lag_7_diff'] = df['Sales_lag_1'] - df['Sales_lag_7']
df['Sales_lag_7_vs_lag_30_diff'] = df['Sales_lag_7'] - df['Sales_lag_30']

df['Promo_Impact'] = df['Discount'] * df['Holiday/Promotion']
df['Price_diff'] = df['Price'] - df['Competitor Pricing']
df['High_Discount'] = (df['Discount'] > 0.2).astype(int)

df.fillna(0, inplace=True)


# Input features and Target
I created input features from the features and I set target="Units Sold"

In [ ]:
#input features and target
target="Units Sold"
input_features=["Sales_lag_1", "Sales_lag_7", "Sales_lag_30", "Rolling_mean_3", "Rolling_std_3",
                'Sales_lag_1_diff', "Sales_lag_1_vs_lag_7_diff", 'Sales_lag_7_vs_lag_30_diff',
                "Price", "Competitor Pricing", 'Price_diff', "Discount", 'High_Discount',
                "Holiday/Promotion", 'Promo_Impact',
                "Month", "Is_Weekend", "Is_Holiday",
                ]

X=df[input_features]
y=df[target]


# Train and Test dataset split
I use date as a splitting mechanism since the dataset is time- series.

In [ ]:
#split the dataset to test and traing dataset based on Date
split_date='2023-10-01'
train=df[df["Date"]<=split_date]
test=df[df["Date"]>split_date]
X_train=train[input_features]
Y_train=train[target]

X_test=test[input_features]
Y_test=test[target]
print(X_train.shape)
print(Y_train.shape)
print(X_test.shape)
print(Y_test.shape)
